In [1]:
import pandas as pd
import sqlite3

# 1. Load your local CSV file
file_path = 'clean_bird_observations_.csv'
df = pd.read_csv(file_path)

# 2. Data Preprocessing to match your Star Schema structure
# Create a unique Date_Key (YYYYMMDD) and Observer_ID for the joins
df['Date_Key'] = pd.to_datetime(df['Date']).dt.strftime('%Y%m%d').astype(int)
df['Observer_ID'] = df['Observer'].astype('category').cat.codes + 1

# Create Dimension Tables
dim_date = df[['Date_Key', 'Year', 'Month']].drop_duplicates().rename(columns={'Year': 'year', 'Month': 'month'})
dim_plot = df[['Plot_Name', 'Location_Type']].drop_duplicates()
dim_species = df[['AOU_Code', 'Scientific_Name', 'PIF_Watchlist_Status']].drop_duplicates()
dim_observer = df[['Observer_ID', 'Observer']].drop_duplicates().rename(columns={'Observer': 'Observer_Name'})

# Create Fact Table
fact_observations = df[[
    'Date_Key', 'Plot_Name', 'AOU_Code', 'Observer_ID', 'Visit',
    'Sex', 'Distance', 'Season', 'Habitat_Source', 'ID_Method', 'Flyover_Observed'
]]

# 3. Setup SQLite Database in memory
conn = sqlite3.connect(':memory:')
dim_date.to_sql('dim_date', conn, index=False)
dim_plot.to_sql('dim_plot', conn, index=False)
dim_species.to_sql('dim_species', conn, index=False)
dim_observer.to_sql('dim_observer', conn, index=False)
fact_observations.to_sql('fact_observations', conn, index=False)

# 4. Define all your SQL Queries (Note: Removed 'dbo.' for SQLite compatibility)
queries = {
    "1A: Seasonal sighting trends": """
        WITH seasonal_trends AS (
            SELECT d.year, d.month, COUNT(*) AS total_sightings
            FROM fact_observations f JOIN dim_date d ON f.Date_Key = d.Date_Key
            GROUP BY d.year, d.month
        )
        SELECT year, month, total_sightings,
            LAG(total_sightings) OVER (PARTITION BY month ORDER BY year) AS prev_year,
            total_sightings - LAG(total_sightings) OVER (PARTITION BY month ORDER BY year) AS yoy_change
        FROM seasonal_trends;
    """,
    "1B: Plot Running Totals": """
        WITH plot_counts AS (
            SELECT p.plot_name, d.Date_Key, COUNT(*) AS daily_count
            FROM fact_observations f
            JOIN dim_plot p ON f.Plot_Name = p.Plot_Name
            JOIN dim_date d ON f.Date_Key = d.Date_Key
            GROUP BY p.plot_name, d.Date_Key
        )
        SELECT plot_name, Date_Key, daily_count,
            SUM(daily_count) OVER (PARTITION BY plot_name ORDER BY Date_Key) AS running_total
        FROM plot_counts LIMIT 15;
    """,
    "2A: Species Diversity by Location": """
        SELECT p.Location_Type, COUNT(DISTINCT s.Scientific_Name) AS unique_species
        FROM fact_observations f
        JOIN dim_species s ON f.AOU_Code = s.AOU_Code
        JOIN dim_plot p ON f.Plot_Name = p.Plot_Name
        GROUP BY p.Location_Type;
    """,
    "2B: Male-to-Female Ratio": """
        WITH sex_counts AS (
            SELECT s.Scientific_Name,
                SUM(CASE WHEN f.Sex = 'Male' THEN 1 ELSE 0 END) AS male_count,
                SUM(CASE WHEN f.Sex = 'Female' THEN 1 ELSE 0 END) AS female_count
            FROM fact_observations f
            JOIN dim_species s ON f.AOU_Code = s.AOU_Code
            WHERE f.Sex IN ('Male', 'Female')
            GROUP BY s.Scientific_Name
        )
        SELECT *, CASE WHEN female_count = 0 THEN NULL ELSE CAST(male_count AS FLOAT) / female_count END AS ratio
        FROM sex_counts WHERE male_count > 0 LIMIT 10;
    """,
    "4A: Rank ID Method per Location": """
        WITH method_counts AS (
            SELECT p.Location_Type, f.ID_Method, COUNT(*) AS cnt
            FROM fact_observations f
            JOIN dim_plot p ON f.Plot_Name = p.Plot_Name
            GROUP BY p.Location_Type, f.ID_Method
        ),
        ranked AS (
            SELECT *, RANK() OVER (PARTITION BY Location_Type ORDER BY cnt DESC) AS rnk
            FROM method_counts
        )
        SELECT * FROM ranked WHERE rnk = 1;
    """,
    "5A: High-Activity Observers": """
        WITH observer_counts AS (
            SELECT o.Observer_Name, COUNT(*) AS total_obs
            FROM fact_observations f JOIN dim_observer o ON f.Observer_ID = o.Observer_ID
            GROUP BY o.Observer_Name
        ),
        observer_with_avg AS (
            SELECT *, AVG(total_obs) OVER () AS avg_obs FROM observer_counts
        )
        SELECT * FROM observer_with_avg WHERE total_obs > avg_obs;
    """,
    "6: Watchlist Species": """
        WITH watchlist AS (
            SELECT s.Scientific_Name, s.PIF_Watchlist_Status, COUNT(*) AS sightings
            FROM fact_observations f JOIN dim_species s ON f.AOU_Code = s.AOU_Code
            WHERE s.PIF_Watchlist_Status = 1
            GROUP BY s.Scientific_Name, s.PIF_Watchlist_Status
        )
        SELECT *, RANK() OVER (ORDER BY sightings DESC) AS rnk FROM watchlist;
    """,
    "7: Flattened Temporal Heatmap": """
        SELECT d.Year, d.Month, COUNT(*) AS total_sightings, COUNT(DISTINCT s.Scientific_Name) AS species_count
        FROM fact_observations f
        JOIN dim_date d ON f.Date_Key = d.Date_Key
        JOIN dim_species s ON f.AOU_Code = s.AOU_Code
        GROUP BY d.Year, d.Month ORDER BY d.Year, d.Month;
    """
}

# 5. Run and Print Results
for title, sql in queries.items():
    print(f"\n{'='*50}\n{title}\n{'='*50}")
    result_df = pd.read_sql_query(sql, conn)
    print(result_df)


1A: Seasonal sighting trends
   year  month  total_sightings prev_year yoy_change
0  2018      5             4505      None       None
1  2018      6             5766      None       None
2  2018      7             3993      None       None

1B: Plot Running Totals
    plot_name  Date_Key  daily_count  running_total
0   ANTI-0007  20180523           16             16
1   ANTI-0007  20180625           14             30
2   ANTI-0007  20180716           11             41
3   ANTI-0008  20180523           17             17
4   ANTI-0008  20180625           12             29
5   ANTI-0008  20180716           10             39
6   ANTI-0009  20180523           15             15
7   ANTI-0009  20180625           14             29
8   ANTI-0009  20180716           11             40
9   ANTI-0015  20180523           18             18
10  ANTI-0015  20180625           13             31
11  ANTI-0015  20180716            8             39
12  ANTI-0016  20180523           16             16
13  A